In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import DATA, split_data
sns.set_theme(style="whitegrid", palette="muted")
split_data()

In [ ]:
train_df = pd.read_csv(DATA / "train.csv")
train_df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Raw Selling Price (in Lakhs for readable axis)
sns.histplot(train_df['selling_price'] / 1e5, bins=50, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("1A. Raw Selling Price Distribution (Right-Skewed)")
axes[0].set_xlabel("Selling Price (₹ Lakhs)")

# Right plot: Log-Transformed Selling Price
sns.histplot(np.log1p(train_df['selling_price']), bins=50, kde=True, ax=axes[1], color="seagreen")
axes[1].set_title("1B. Log-Transformed Selling Price (Near-Normal)")
axes[1].set_xlabel("Log(Selling Price)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 2: Car Age vs Average Selling Price
sns.lineplot(data=train_df, x='car_age', y=train_df['selling_price'] / 1e5, marker="o", ax=axes[0], color="crimson")
axes[0].set_title("2. Car Age vs. Selling Price (Depreciation Curve)")
axes[0].set_xlabel("Car Age (Years since 2020)")
axes[0].set_ylabel("Selling Price (₹ Lakhs)")

# Chart 3: Kilometers Driven vs Selling Price (filtering out extreme >5 Lakh km outlier for visual clarity)
subset_km = train_df[train_df['km_driven'] <= 300000]
sns.scatterplot(data=subset_km, x='km_driven', y=subset_km['selling_price'] / 1e5, alpha=0.4, ax=axes[1], color="orange")
axes[1].set_title("3. Kilometers Driven vs. Selling Price (Up to 3L km)")
axes[1].set_xlabel("Kilometers Driven")
axes[1].set_ylabel("Selling Price (₹ Lakhs)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 4: Fuel Type vs Selling Price
sns.barplot(data=train_df, x='fuel', y=train_df['selling_price'] / 1e5, ax=axes[0], palette="viridis")
axes[0].set_title("4. Average Selling Price by Fuel Type")
axes[0].set_xlabel("Fuel Type")
axes[0].set_ylabel("Average Price (₹ Lakhs)")

# Chart 5: Transmission Type vs Selling Price
sns.barplot(data=train_df, x='transmission', y=train_df['selling_price'] / 1e5, ax=axes[1], palette="magma")
axes[1].set_title("5. Average Selling Price by Transmission")
axes[1].set_xlabel("Transmission Type")
axes[1].set_ylabel("Average Price (₹ Lakhs)")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Calculate average price per brand (in Lakhs) and sort descending
brand_avg = (train_df.groupby('brand')['selling_price'].mean() / 1e5).sort_values(ascending=False).head(15)

sns.barplot(x=brand_avg.values, y=brand_avg.index, palette="coolwarm")
plt.title("6. Top 15 Car Brands by Average Resale Price (Training Set)")
plt.xlabel("Average Selling Price (₹ Lakhs)")
plt.ylabel("Car Brand")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))

# Select numeric columns for correlation
num_cols = ['selling_price', 'car_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']
corr_matrix = train_df[num_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, linewidths=0.5)
plt.title("7. Feature Correlation Heatmap (Training Data)")
plt.tight_layout()
plt.show()

### Day 3 Exploratory Data Analysis (EDA) Findings
1. **Target Skewness (`selling_price`):** Raw prices are heavily right-skewed due to luxury vehicles reaching ₹1 Crore. Applying `np.log1p()` transforms the target into a clean bell-shaped distribution, confirming the need for `TransformedTargetRegressor`.
2. **Primary Positive Driver (`max_power` & `engine`):** `max_power` has the strongest positive correlation with `selling_price` (~0.69), followed by `engine` CC (~0.45). More powerful engines belong to higher-segment and luxury cars.
3. **Primary Negative Driver (`car_age` & `km_driven`):** `car_age` shows a strong negative relationship with price—cars lose value rapidly in their first 5–8 years. Higher `km_driven` also lowers resale value.
4. **Fuel & Transmission Premiums:** `Diesel` cars command higher average resale prices than `Petrol`, `CNG`, and `LPG` (as SUVs and larger sedans are predominantly Diesel). `Automatic` transmission vehicles have a significantly higher average resale value than `Manual` cars because automatics are more common in premium/newer models.
5. **Brand Segmentation (`brand`):** Luxury brands (`Volvo`, `BMW`, `Mercedes-Benz`, `Audi`, `Jaguar`, `Land Rover`) average ₹25–₹50+ Lakhs, whereas mass-market brands (`Maruti`, `Hyundai`, `Tata`) average ₹2–₹6 Lakhs, proving `brand` is a critical categorical predictor.